# 03. FT-Transformer v2 — TE Ablation

기존 최고 성능 FT-Transformer v2를 보존하면서, **39개 exact-value multiclass Target Encoding(TE)**을 더 효율적으로 구성할 수 있는지 검증하는 노트북입니다.

핵심 원칙은 다음과 같습니다.

- 단순한 전체 Pearson correlation으로 TE를 자르지 않습니다. 3-class 명목형 타깃에는 클래스별 one-vs-rest correlation을 진단값으로만 사용합니다.
- TE 관련 통계는 outer training data 내부의 cross-fitted TE에서만 계산합니다.
- 개별 TE 컬럼이 아니라 **원본 변수에서 파생된 class TE 묶음**을 하나의 그룹으로 평가합니다.
- 먼저 저비용 고정 holdout LOFO screening을 수행하고, 살아남은 후보만 정식 7-fold로 확인합니다.

기존 결과 기준:

- 원본 13개 + TE 39개 = 52개 입력
- FT-Transformer 7-fold × fold당 4-member ensemble × 16 epochs
- fixed inverse-prior 제출: Private 0.95066 / Public 0.95054


## 1. 실행 모드

기본값은 `candidate-screen`입니다. 이미 완료한 LOFO 결과를 바탕으로 52·46·39·37개 후보를 같은 조건에서 직접 비교합니다.

- `te_probability_sum_audit.csv`: 각 원본 변수의 3-class TE 합이 1인지 검사
- `te_classwise_correlation.csv`: 내부 OOF TE와 각 클래스 one-vs-rest 타깃의 correlation
- `lofo_screening.csv`: `at-risk` TE를 제거한 26개 TE 기준 그룹 LOFO 결과
- `lofo_recommended_groups.json`: 정식 검증 후보 그룹 목록
- `candidate_screening.csv`: 52·46·39·37개 입력 후보의 직접 비교 결과

실행 모드:

| 모드 | 용도 |
|---|---|
| `audit-only` | TE 중복·correlation만 검사하고 모델은 학습하지 않음 |
| `lofo-screen` | 고정 20% validation, 1-member, 4 epochs로 13개 그룹을 빠르게 선별 |
| `candidate-screen` | 기존 52개, 안전 축소 46개, 공격적 축소 39개, BMI 제거 37개를 같은 holdout에서 비교 |
| `full-compare` | 기존 52개 입력과 `at-risk` TE 제거 39개 입력을 동일 7-fold로 정식 비교 |
| `selected-full` | LOFO 이후 직접 지정한 TE 그룹만 정식 7-fold 학습 |

`full-compare`는 기존 대비 약 2배의 FTT 학습이 필요합니다. screening 결과를 먼저 확인한 뒤 실행하는 것을 권장합니다.


In [ ]:
# Kaggle 실행 설정
import sys

RUN_MODE = "candidate-screen"  # audit-only | lofo-screen | candidate-screen | full-compare | selected-full

# selected-full일 때만 수정합니다. 예: "sleep_hours,stress_level,bmi"
SELECTED_TE_GROUPS = ""

sys.argv = [
    "03_ft_transformer_v2_te_ablation.py",
    "--data-dir", "/kaggle/input/competitions/playground-series-s6e7",
    "--output-dir", "/kaggle/working/ftt_te_ablation_outputs",
    "--run-mode", RUN_MODE,
    "--n-splits", "7",
    "--n-ens", "4",
    "--n-epochs", "16",
    "--screen-epochs", "4",
    "--screen-n-ens", "1",
    "--screen-valid-size", "0.20",
    "--reference-class", "at-risk",
    "--include-te-groups", SELECTED_TE_GROUPS,
    "--device", "cuda",
    "--resume",
]


In [ ]:
# Kaggle 기본 이미지에 없는 프로젝트 의존성을 현재 커널에 설치합니다.
import importlib.util
import subprocess

required_packages = ["catstat", "masamlp"]
missing_packages = [
    package
    for package in required_packages
    if importlib.util.find_spec(package) is None
]
if missing_packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages]
    )

import catstat
import masamlp

print("catstat:", getattr(catstat, "__version__", "unknown"))
print("masamlp:", getattr(masamlp, "__version__", "unknown"))


## 2. 공통 유틸리티와 인자

실험명·TE 그룹·제거 class를 cache 설정에 포함해 서로 다른 실험의 fold cache가 섞이지 않도록 합니다.


In [ ]:
import argparse
import gc
import hashlib
import json
import os
import re
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder


TARGET = "health_condition"
ID_COL = "id"
OUTER_SEED = 42
TE_SEED = 42
MODEL_SEED = 42
TE_CV = 5


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="FTT v2 grouped TE ablation")
    parser.add_argument("--data-dir", type=Path, default=None)
    parser.add_argument("--output-dir", type=Path, default=None)
    parser.add_argument(
        "--run-mode",
        choices=[
            "audit-only",
            "lofo-screen",
            "candidate-screen",
            "full-compare",
            "selected-full",
        ],
        default="lofo-screen",
    )
    parser.add_argument("--n-splits", type=int, default=7)
    parser.add_argument("--n-ens", type=int, default=4)
    parser.add_argument("--n-epochs", type=int, default=16)
    parser.add_argument("--screen-epochs", type=int, default=4)
    parser.add_argument("--screen-n-ens", type=int, default=1)
    parser.add_argument("--screen-valid-size", type=float, default=0.20)
    parser.add_argument("--reference-class", default="at-risk")
    parser.add_argument("--include-te-groups", default="")
    parser.add_argument("--resume", action="store_true")
    parser.add_argument("--device", default="cuda")
    return parser.parse_args()


def resolve_data_dir(requested: Path | None) -> Path:
    candidates = []
    if requested is not None:
        candidates.append(requested)
    env_path = os.environ.get("S6E7_DATA_DIR")
    if env_path:
        candidates.append(Path(env_path))
    script_root = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    candidates.extend([
        Path("/kaggle/input/playground-series-s6e7"),
        script_root / "downloads" / "DScover 가이드 프로젝트",
        Path.cwd() / "downloads" / "DScover 가이드 프로젝트",
    ])
    for candidate in candidates:
        if all((candidate / name).exists() for name in ["train.csv", "test.csv"]):
            return candidate.resolve()
    raise FileNotFoundError("Could not locate train.csv/test.csv. Pass --data-dir or set S6E7_DATA_DIR.")


def resolve_output_dir(requested: Path | None) -> Path:
    if requested is not None:
        return requested.resolve()
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working/ftt_te_ablation_outputs")
    return Path.cwd() / "ftt_te_ablation_outputs"


def safe_name(value: str) -> str:
    return re.sub(r"[^0-9A-Za-z가-힣_-]+", "_", str(value)).strip("_")


def array_sha256(values: np.ndarray) -> str:
    contiguous = np.ascontiguousarray(values)
    return hashlib.sha256(contiguous.view(np.uint8)).hexdigest()


def fast_balanced_accuracy(y_true: np.ndarray, prediction: np.ndarray) -> float:
    counts = np.bincount(y_true, minlength=3)
    correct = np.bincount(y_true[y_true == prediction], minlength=3)
    return float(np.mean(correct / counts))


def recalls(y_true: np.ndarray, prediction: np.ndarray) -> list[float]:
    return [float(np.mean(prediction[y_true == c] == c)) for c in range(3)]


def normalize_probability(proba: np.ndarray) -> np.ndarray:
    proba = np.asarray(proba, dtype=np.float64)
    proba = np.clip(proba, 1e-12, None)
    return proba / proba.sum(axis=1, keepdims=True)


@dataclass(frozen=True)
class ExperimentSpec:
    name: str
    te_groups: tuple[str, ...]
    drop_class: str | None
    # None means drop the class channel from every selected TE group.
    # A tuple limits the drop to those source-feature groups only.
    drop_class_groups: tuple[str, ...] | None = None


## 3. 명시적 TE 그룹 생성과 진단

기존 노트북은 TE 컬럼 이름이 `exact_te_00~38`이라 원본 변수·클래스 매핑을 확인하기 어려웠습니다. 여기서는 각 원본 변수를 독립적으로 encoding하여 다음 이름을 부여합니다.

`te__원본변수__클래스`

각 encoder는 같은 internal 5-fold seed를 사용합니다. 이로써 기존의 독립 exact-value unit 구조를 유지하면서 그룹 선택이 명시적으로 가능해집니다.


In [ ]:
def te_column_name(feature: str, class_name: str) -> str:
    return f"te__{safe_name(feature)}__{safe_name(class_name)}"


def build_exact_value_te(
    fit_raw: pd.DataFrame,
    valid_raw: pd.DataFrame,
    test_raw: pd.DataFrame | None,
    y_fit: np.ndarray,
    feature_columns: list[str],
    class_names: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame | None, pd.DataFrame]:
    from catstat import TargetEncoder

    fit_blocks, valid_blocks, test_blocks = [], [], []
    mapping_rows = []

    # One encoder per source feature makes the feature/class mapping explicit.
    # All encoders use identical cross-fitting settings and are fit only on the
    # current outer-training rows.
    for feature in feature_columns:
        encoder = TargetEncoder(
            cols=[feature],
            stats=("mean",),
            target_type="multiclass",
            smooth="auto",
            numeric="direct",
            cv=TE_CV,
            random_state=TE_SEED,
            output="numpy",
        )
        fit_block = np.asarray(
            encoder.fit_transform(fit_raw[[feature]], y_fit), dtype=np.float32
        )
        valid_block = np.asarray(
            encoder.transform(valid_raw[[feature]]), dtype=np.float32
        )
        test_block = None
        if test_raw is not None:
            test_block = np.asarray(
                encoder.transform(test_raw[[feature]]), dtype=np.float32
            )

        expected_shape = (len(fit_raw), len(class_names))
        if fit_block.shape != expected_shape:
            raise ValueError(
                f"Unexpected TE shape for {feature}: {fit_block.shape}; expected {expected_shape}"
            )
        names = [te_column_name(feature, class_name) for class_name in class_names]
        fit_blocks.append(pd.DataFrame(fit_block, columns=names))
        valid_blocks.append(pd.DataFrame(valid_block, columns=names))
        if test_block is not None:
            test_blocks.append(pd.DataFrame(test_block, columns=names))
        mapping_rows.extend(
            {
                "source_feature": feature,
                "class_index": class_index,
                "class_name": class_name,
                "te_column": names[class_index],
            }
            for class_index, class_name in enumerate(class_names)
        )

    fit_te = pd.concat(fit_blocks, axis=1)
    valid_te = pd.concat(valid_blocks, axis=1)
    test_te = pd.concat(test_blocks, axis=1) if test_blocks else None
    mapping = pd.DataFrame(mapping_rows)
    return fit_te, valid_te, test_te, mapping


def te_probability_sum_audit(
    te_frame: pd.DataFrame,
    feature_columns: list[str],
    class_names: list[str],
    split_name: str,
) -> pd.DataFrame:
    rows = []
    for feature in feature_columns:
        columns = [te_column_name(feature, class_name) for class_name in class_names]
        values = te_frame[columns].to_numpy(dtype=np.float64)
        error = np.abs(values.sum(axis=1) - 1.0)
        rows.append({
            "split": split_name,
            "source_feature": feature,
            "mean_abs_sum_error": float(error.mean()),
            "p99_abs_sum_error": float(np.quantile(error, 0.99)),
            "max_abs_sum_error": float(error.max()),
            "near_exact_sum": bool(error.max() <= 1e-5),
        })
    return pd.DataFrame(rows)


def te_classwise_correlation(
    fit_te: pd.DataFrame,
    y_fit: np.ndarray,
    feature_columns: list[str],
    class_names: list[str],
) -> pd.DataFrame:
    # fit_te comes from TargetEncoder.fit_transform(), hence it is internally
    # cross-fitted OOF TE rather than an in-sample encoding.
    rows = []
    for feature in feature_columns:
        for class_index, class_name in enumerate(class_names):
            column = te_column_name(feature, class_name)
            x = fit_te[column].to_numpy(dtype=np.float64)
            target_binary = (y_fit == class_index).astype(np.float64)
            if np.std(x) == 0 or np.std(target_binary) == 0:
                corr = np.nan
            else:
                corr = float(np.corrcoef(x, target_binary)[0, 1])
            rows.append({
                "source_feature": feature,
                "te_class": class_name,
                "correlation_one_vs_rest": corr,
                "abs_correlation": abs(corr) if np.isfinite(corr) else np.nan,
            })
    result = pd.DataFrame(rows)
    group_max = result.groupby("source_feature")["abs_correlation"].transform("max")
    result["group_max_abs_correlation"] = group_max
    return result.sort_values(
        ["group_max_abs_correlation", "source_feature", "abs_correlation"],
        ascending=[False, True, False],
    ).reset_index(drop=True)


def model_frame(
    raw: pd.DataFrame,
    te: pd.DataFrame,
    feature_columns: list[str],
    class_names: list[str],
    spec: ExperimentSpec,
) -> pd.DataFrame:
    selected_te = []
    if spec.drop_class is None:
        drop_groups: set[str] = set()
    elif spec.drop_class_groups is None:
        drop_groups = set(spec.te_groups)
    else:
        drop_groups = set(spec.drop_class_groups)
    for feature in spec.te_groups:
        if feature not in feature_columns:
            raise ValueError(f"Unknown TE group: {feature}")
        for class_name in class_names:
            if not (class_name == spec.drop_class and feature in drop_groups):
                selected_te.append(te_column_name(feature, class_name))
    return pd.concat(
        [raw[feature_columns].reset_index(drop=True), te[selected_te].reset_index(drop=True)],
        axis=1,
    )


## 4. FT-Transformer와 decision rule

모델 사양은 기존 v2와 동일합니다. LOFO screening에서만 epoch와 ensemble member 수를 낮춥니다. 모든 feature 실험은 같은 validation 행, seed, prior correction을 사용합니다.


In [ ]:
def make_model(
    categorical_columns: list[str],
    n_ens: int,
    n_epochs: int,
    device: str,
):
    from masamlp import MasaClassifier

    return MasaClassifier(
        model="ft_transformer",
        model_params={"d_block": 128, "n_blocks": 2, "attention_n_heads": 8},
        n_epochs=n_epochs,
        batch_size=4096,
        eval_batch_size=8192,
        learning_rate=1e-3,
        weight_decay=1e-5,
        optimizer="adamw",
        lr_scheduler="cosine",
        num_embedding="plr-lite",
        numeric_scaler="quantile",
        categorical_features=categorical_columns,
        cat_encoding="embedding",
        n_ens=n_ens,
        ens_mode="loop",
        early_stopping_rounds=None,
        class_weight=None,
        device=device,
        amp="auto",
        verbose=1,
        random_state=MODEL_SEED,
    )


def reorder_model_probability(model, proba: np.ndarray) -> np.ndarray:
    model_classes = np.asarray(model.classes_)
    positions = []
    for label in np.arange(3):
        match = np.flatnonzero(model_classes == label)
        if len(match) != 1:
            raise ValueError(f"Unexpected model classes: {model_classes.tolist()}")
        positions.append(int(match[0]))
    return normalize_probability(proba[:, positions]).astype(np.float32)


def prior_prediction(proba: np.ndarray, prior: np.ndarray, beta: float = 1.0) -> np.ndarray:
    return np.argmax(proba / np.power(prior, beta), axis=1)


def tune_prior_beta(
    y_true: np.ndarray,
    proba: np.ndarray,
    prior: np.ndarray,
    seed: int,
) -> tuple[float, float]:
    def objective(vector: np.ndarray) -> float:
        return -fast_balanced_accuracy(
            y_true, prior_prediction(proba, prior, beta=float(vector[0]))
        )

    result = differential_evolution(
        objective,
        bounds=[(0.5, 1.5)],
        seed=seed,
        popsize=10,
        maxiter=18,
        polish=False,
        updating="immediate",
        workers=1,
    )
    return float(result.x[0]), -float(result.fun)


def tune_multipliers(
    y_true: np.ndarray,
    proba: np.ndarray,
    prior: np.ndarray,
    seed: int,
) -> tuple[np.ndarray, float]:
    initial = np.log((1.0 / prior) / (1.0 / prior)[0])[1:]

    def objective(log_ratios: np.ndarray) -> float:
        multipliers = np.exp(np.r_[0.0, log_ratios])
        return -fast_balanced_accuracy(y_true, np.argmax(proba * multipliers, axis=1))

    result = differential_evolution(
        objective,
        bounds=[
            (float(initial[0] - 0.75), float(initial[0] + 0.75)),
            (float(initial[1] - 0.75), float(initial[1] + 0.75)),
        ],
        seed=seed,
        popsize=7,
        maxiter=18,
        polish=False,
        updating="immediate",
        workers=1,
    )
    multipliers = np.exp(np.r_[0.0, result.x])
    return multipliers, -float(result.fun)


def crossfit_decisions(
    y: np.ndarray,
    oof: np.ndarray,
    fold_id: np.ndarray,
    n_splits: int,
) -> tuple[np.ndarray, np.ndarray, list[dict]]:
    beta_prediction = np.empty(len(y), dtype=np.int8)
    multiplier_prediction = np.empty(len(y), dtype=np.int8)
    rows = []
    for heldout in range(n_splits):
        meta_fit = fold_id != heldout
        meta_valid = fold_id == heldout
        prior = np.bincount(y[meta_fit], minlength=3).astype(np.float64)
        prior /= prior.sum()
        beta, beta_train_score = tune_prior_beta(
            y[meta_fit], oof[meta_fit], prior, 7000 + heldout
        )
        multipliers, multiplier_train_score = tune_multipliers(
            y[meta_fit], oof[meta_fit], prior, 8000 + heldout
        )
        beta_prediction[meta_valid] = prior_prediction(oof[meta_valid], prior, beta)
        multiplier_prediction[meta_valid] = np.argmax(
            oof[meta_valid] * multipliers, axis=1
        )
        rows.append({
            "heldout_fold": heldout + 1,
            "prior": prior.tolist(),
            "beta": beta,
            "beta_meta_fit_score": beta_train_score,
            "beta_heldout_score": fast_balanced_accuracy(
                y[meta_valid], beta_prediction[meta_valid]
            ),
            "multipliers": multipliers.tolist(),
            "multiplier_meta_fit_score": multiplier_train_score,
            "multiplier_heldout_score": fast_balanced_accuracy(
                y[meta_valid], multiplier_prediction[meta_valid]
            ),
        })
    return beta_prediction, multiplier_prediction, rows


def metric_row(name: str, y: np.ndarray, prediction: np.ndarray) -> dict:
    class_recalls = recalls(y, prediction)
    return {
        "candidate": name,
        "balanced_accuracy": fast_balanced_accuracy(y, prediction),
        "at_risk_recall": class_recalls[0],
        "fit_recall": class_recalls[1],
        "unhealthy_recall": class_recalls[2],
    }


def save_submission(
    sample: pd.DataFrame,
    label_encoder: LabelEncoder,
    prediction: np.ndarray,
    path: Path,
) -> None:
    submission = sample.copy()
    submission[TARGET] = label_encoder.inverse_transform(prediction)
    if list(submission.columns) != [ID_COL, TARGET]:
        raise ValueError(f"Unexpected submission columns: {submission.columns.tolist()}")
    if submission[ID_COL].duplicated().any() or submission[TARGET].isna().any():
        raise ValueError("Invalid submission: duplicate ID or missing prediction")
    submission.to_csv(path, index=False)


## 5. 저비용 그룹 LOFO screening

먼저 `at-risk` TE를 제거해 원본 13 + TE 26 구성을 기준으로 삼습니다. 이후 원본 feature는 유지한 채, 각 원본 변수에서 파생된 두 TE만 한 그룹씩 제거합니다.

`lofo_importance = baseline inverse-prior BA − 해당 그룹 제거 BA`

- 양수: 해당 TE 그룹을 제거하면 성능이 떨어졌으므로 유용할 가능성
- 0 부근: 영향이 작음
- 음수: 제거했을 때 오히려 개선되어 잡음일 가능성

screening은 후보 생성용이며 최종 결론이 아닙니다. 최종 선택은 반드시 정식 7-fold에서 확인합니다.


In [ ]:
def save_te_diagnostics(
    fit_te: pd.DataFrame,
    valid_te: pd.DataFrame,
    y_fit: np.ndarray,
    feature_columns: list[str],
    class_names: list[str],
    mapping: pd.DataFrame,
    output_dir: Path,
) -> pd.DataFrame:
    audit = pd.concat([
        te_probability_sum_audit(fit_te, feature_columns, class_names, "internal_oof_fit"),
        te_probability_sum_audit(valid_te, feature_columns, class_names, "outer_valid"),
    ], ignore_index=True)
    correlation = te_classwise_correlation(
        fit_te, y_fit, feature_columns, class_names
    )
    audit.to_csv(output_dir / "te_probability_sum_audit.csv", index=False)
    correlation.to_csv(output_dir / "te_classwise_correlation.csv", index=False)
    mapping.to_csv(output_dir / "te_column_mapping.csv", index=False)
    print("\nTE probability-sum audit")
    print(audit.to_string(index=False))
    print("\nClasswise OOF TE correlation (diagnostic only)")
    print(correlation.to_string(index=False))
    return audit


def run_screening(
    args: argparse.Namespace,
    raw_train: pd.DataFrame,
    y: np.ndarray,
    feature_columns: list[str],
    categorical_columns: list[str],
    class_names: list[str],
    output_dir: Path,
) -> None:
    import torch

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=args.screen_valid_size,
        random_state=OUTER_SEED,
    )
    fit_idx, valid_idx = next(splitter.split(raw_train, y))
    fit_raw = raw_train.iloc[fit_idx].reset_index(drop=True)
    valid_raw = raw_train.iloc[valid_idx].reset_index(drop=True)
    y_fit, y_valid = y[fit_idx], y[valid_idx]

    fit_te, valid_te, _, mapping = build_exact_value_te(
        fit_raw,
        valid_raw,
        None,
        y_fit,
        feature_columns,
        class_names,
    )
    audit = save_te_diagnostics(
        fit_te,
        valid_te,
        y_fit,
        feature_columns,
        class_names,
        mapping,
        output_dir,
    )
    if args.run_mode == "audit-only":
        print("audit-only complete; no FTT model was trained.")
        return

    if args.reference_class not in class_names:
        raise ValueError(
            f"reference-class={args.reference_class!r} not in classes={class_names}"
        )

    all_groups = tuple(feature_columns)
    if args.run_mode == "candidate-screen":
        if "bmi" not in feature_columns:
            raise ValueError("candidate-screen expects the bmi source feature")
        specs = [
            ExperimentSpec("baseline52", all_groups, None),
            ExperimentSpec(
                "safe46_categorical_reference_dropped",
                all_groups,
                args.reference_class,
                tuple(categorical_columns),
            ),
            ExperimentSpec("aggressive39_all_reference_dropped", all_groups, args.reference_class),
            ExperimentSpec(
                "aggressive37_without_bmi",
                tuple(feature for feature in feature_columns if feature != "bmi"),
                args.reference_class,
            ),
        ]
    else:
        specs = [
            ExperimentSpec("te26_all_groups", all_groups, args.reference_class)
        ] + [
            ExperimentSpec(
                f"without_{safe_name(excluded)}",
                tuple(feature for feature in feature_columns if feature != excluded),
                args.reference_class,
            )
            for excluded in feature_columns
        ]

    prior = np.bincount(y_fit, minlength=3).astype(np.float64)
    prior /= prior.sum()
    rows = []
    for spec in specs:
        started = time.time()
        fit_frame = model_frame(
            fit_raw, fit_te, feature_columns, class_names, spec
        )
        valid_frame = model_frame(
            valid_raw, valid_te, feature_columns, class_names, spec
        )
        model = make_model(
            categorical_columns,
            n_ens=args.screen_n_ens,
            n_epochs=args.screen_epochs,
            device=args.device,
        )
        model.fit(fit_frame, y_fit)
        proba = reorder_model_probability(model, model.predict_proba(valid_frame))
        raw_pred = proba.argmax(axis=1)
        inverse_pred = prior_prediction(proba, prior, beta=1.0)
        excluded_candidates = [
            feature for feature in feature_columns if feature not in spec.te_groups
        ]
        excluded = "" if not excluded_candidates else excluded_candidates[0]
        inv_recalls = recalls(y_valid, inverse_pred)
        row = {
            "experiment": spec.name,
            "excluded_te_group": excluded,
            "raw_feature_count": len(feature_columns),
            "te_feature_count": fit_frame.shape[1] - len(feature_columns),
            "total_feature_count": fit_frame.shape[1],
            "raw_balanced_accuracy": fast_balanced_accuracy(y_valid, raw_pred),
            "inverse_prior_balanced_accuracy": fast_balanced_accuracy(y_valid, inverse_pred),
            "inverse_prior_at_risk_recall": inv_recalls[0],
            "inverse_prior_fit_recall": inv_recalls[1],
            "inverse_prior_unhealthy_recall": inv_recalls[2],
            "logloss": log_loss(y_valid, proba, labels=np.arange(3)),
            "seconds": time.time() - started,
        }
        rows.append(row)
        print(json.dumps(row), flush=True)
        del model, fit_frame, valid_frame, proba
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    result = pd.DataFrame(rows)
    baseline_name = "baseline52" if args.run_mode == "candidate-screen" else "te26_all_groups"
    baseline_ba = float(
        result.loc[
            result["experiment"] == baseline_name,
            "inverse_prior_balanced_accuracy",
        ].iloc[0]
    )
    if args.run_mode == "candidate-screen":
        result["ba_delta_vs_baseline52"] = (
            result["inverse_prior_balanced_accuracy"] - baseline_ba
        )
        result = result.sort_values(
            ["inverse_prior_balanced_accuracy", "logloss"],
            ascending=[False, True],
        ).reset_index(drop=True)
        result.to_csv(output_dir / "candidate_screening.csv", index=False)
        print("\nCandidate screening result")
        print(result.to_string(index=False))
        print("\nRun full CV only for the best two candidates.")
        return

    result["lofo_importance"] = np.where(
        result["experiment"] == "te26_all_groups",
        np.nan,
        baseline_ba - result["inverse_prior_balanced_accuracy"],
    )
    result = result.sort_values(
        ["lofo_importance", "experiment"], ascending=[False, True], na_position="first"
    ).reset_index(drop=True)
    result.to_csv(output_dir / "lofo_screening.csv", index=False)

    useful = result.loc[result["lofo_importance"] > 0, "excluded_te_group"].tolist()
    ranked_groups = result.loc[
        result["excluded_te_group"] != "", "excluded_te_group"
    ].tolist()
    recommendation = {
        "screening_only": True,
        "rule": "Groups with positive LOFO importance; confirm candidate subsets with full CV.",
        "reference_class_removed": args.reference_class,
        "baseline_inverse_prior_balanced_accuracy": baseline_ba,
        "positive_importance_groups": useful,
        "ranked_groups": ranked_groups,
        "top_6_groups": ranked_groups[:6],
        "top_8_groups": ranked_groups[:8],
        "top_10_groups": ranked_groups[:10],
        "screen_epochs": args.screen_epochs,
        "screen_n_ens": args.screen_n_ens,
        "screen_valid_size": args.screen_valid_size,
        "all_te_probability_sums_near_one": bool(audit["near_exact_sum"].all()),
    }
    (output_dir / "lofo_recommended_groups.json").write_text(
        json.dumps(recommendation, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print("\nLOFO screening result")
    print(result.to_string(index=False))
    print("\nScreening candidate groups (must be confirmed with full CV):")
    print(useful)


## 6. 정식 CV 비교

`full-compare`는 같은 7-fold에서 다음 두 구성을 비교합니다.

- `baseline52`: 원본 13 + 모든 class TE 39
- `drop_at_risk39`: 원본 13 + `fit`/`unhealthy` TE 26

`selected-full`은 `SELECTED_TE_GROUPS`로 지정한 그룹만 사용하며 기본적으로 `at-risk` TE를 제거합니다. 각 실험은 자체 cache·OOF·submission을 가지므로 중단 후 `--resume`으로 이어갈 수 있습니다.


In [ ]:
def fold_cache_matches(
    cached,
    valid_idx: np.ndarray,
    train_ids: np.ndarray,
    test_ids: np.ndarray,
    config_json: str,
) -> bool:
    return (
        np.array_equal(cached["valid_idx"], valid_idx)
        and np.array_equal(cached["valid_ids"], train_ids[valid_idx])
        and np.array_equal(cached["test_ids"], test_ids)
        and str(cached["config_json"].item()) == config_json
    )


def full_experiment_specs(
    args: argparse.Namespace,
    feature_columns: list[str],
) -> list[ExperimentSpec]:
    all_groups = tuple(feature_columns)
    if args.run_mode == "full-compare":
        return [
            ExperimentSpec("baseline52", all_groups, None),
            ExperimentSpec("drop_at_risk39", all_groups, args.reference_class),
        ]
    selected = tuple(
        item.strip() for item in args.include_te_groups.split(",") if item.strip()
    )
    if not selected:
        raise ValueError(
            "selected-full requires --include-te-groups, e.g. sleep_hours,stress_level,bmi"
        )
    unknown = sorted(set(selected) - set(feature_columns))
    if unknown:
        raise ValueError(f"Unknown selected TE groups: {unknown}")
    return [
        ExperimentSpec(
            f"selected_{len(selected)}_groups",
            selected,
            args.reference_class,
        )
    ]


def evaluate_and_save_full_experiment(
    spec: ExperimentSpec,
    args: argparse.Namespace,
    y: np.ndarray,
    fold_id: np.ndarray,
    oof: np.ndarray,
    test_proba: np.ndarray,
    prior: np.ndarray,
    sample: pd.DataFrame,
    label_encoder: LabelEncoder,
    classes: np.ndarray,
    train_ids: np.ndarray,
    test_ids: np.ndarray,
    fold_rows: list[dict],
    experiment_dir: Path,
    config: dict,
) -> list[dict]:
    raw_prediction = oof.argmax(axis=1)
    inverse_prediction = prior_prediction(oof, prior, beta=1.0)
    beta_crossfit_prediction, multiplier_crossfit_prediction, post_rows = crossfit_decisions(
        y, oof, fold_id, args.n_splits
    )
    full_beta, full_beta_score = tune_prior_beta(y, oof, prior, seed=9001)
    full_multipliers, full_multiplier_score = tune_multipliers(
        y, oof, prior, seed=9002
    )

    comparison = pd.DataFrame([
        metric_row(f"{spec.name}_raw_argmax", y, raw_prediction),
        metric_row(f"{spec.name}_fixed_inverse_prior", y, inverse_prediction),
        metric_row(f"{spec.name}_prior_beta_crossfit", y, beta_crossfit_prediction),
        metric_row(
            f"{spec.name}_unrestricted_multiplier_crossfit",
            y,
            multiplier_crossfit_prediction,
        ),
    ])
    comparison.insert(0, "experiment", spec.name)
    comparison.to_csv(experiment_dir / "comparison.csv", index=False)
    pd.DataFrame(fold_rows).to_csv(experiment_dir / "fold_metrics.csv", index=False)
    pd.DataFrame(post_rows).to_json(
        experiment_dir / "postprocess_folds.json", orient="records", indent=2
    )
    np.savez_compressed(
        experiment_dir / "probabilities.npz",
        oof=oof,
        test=test_proba,
        y=y,
        fold_id=fold_id,
        classes=classes,
        train_ids=train_ids,
        test_ids=test_ids,
        class_prior=prior,
    )
    save_submission(
        sample,
        label_encoder,
        test_proba.argmax(axis=1),
        experiment_dir / "submission_raw.csv",
    )
    save_submission(
        sample,
        label_encoder,
        prior_prediction(test_proba, prior, beta=1.0),
        experiment_dir / "submission_inverse_prior.csv",
    )
    save_submission(
        sample,
        label_encoder,
        prior_prediction(test_proba, prior, beta=full_beta),
        experiment_dir / "submission_prior_beta.csv",
    )
    save_submission(
        sample,
        label_encoder,
        np.argmax(test_proba * full_multipliers, axis=1),
        experiment_dir / "submission_multiplier.csv",
    )
    summary = {
        "config": config,
        "comparison": comparison.to_dict(orient="records"),
        "full_oof_beta_for_test": full_beta,
        "full_oof_beta_score_optimistic": full_beta_score,
        "full_oof_multipliers_for_test": full_multipliers.tolist(),
        "full_oof_multiplier_score_optimistic": full_multiplier_score,
        "oof_sha256": array_sha256(oof),
        "test_proba_sha256": array_sha256(test_proba),
    }
    (experiment_dir / "summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    return comparison.to_dict(orient="records")


def run_full_cv(
    args: argparse.Namespace,
    train: pd.DataFrame,
    test: pd.DataFrame,
    sample: pd.DataFrame,
    raw_train: pd.DataFrame,
    raw_test: pd.DataFrame,
    y: np.ndarray,
    label_encoder: LabelEncoder,
    feature_columns: list[str],
    categorical_columns: list[str],
    classes: np.ndarray,
    output_dir: Path,
) -> None:
    import catstat
    import masamlp
    import torch

    specs = full_experiment_specs(args, feature_columns)
    if args.reference_class not in classes.tolist():
        raise ValueError(
            f"reference-class={args.reference_class!r} not in classes={classes.tolist()}"
        )

    train_ids = train[ID_COL].to_numpy()
    test_ids = test[ID_COL].to_numpy()
    fold_id = np.full(len(train), -1, dtype=np.int8)
    state = {
        spec.name: {
            "oof": np.zeros((len(train), 3), dtype=np.float32),
            "test_sum": np.zeros((len(test), 3), dtype=np.float64),
            "fold_rows": [],
        }
        for spec in specs
    }
    outer = StratifiedKFold(
        n_splits=args.n_splits, shuffle=True, random_state=OUTER_SEED
    )
    folds = list(outer.split(raw_train, y))
    class_names = classes.astype(str).tolist()

    for fold, (fit_idx, valid_idx) in enumerate(folds, start=1):
        fold_id[valid_idx] = fold - 1
        fit_raw = raw_train.iloc[fit_idx].reset_index(drop=True)
        valid_raw = raw_train.iloc[valid_idx].reset_index(drop=True)
        test_raw = raw_test.reset_index(drop=True)
        fit_te, valid_te, test_te, mapping = build_exact_value_te(
            fit_raw,
            valid_raw,
            test_raw,
            y[fit_idx],
            feature_columns,
            class_names,
        )
        if fold == 1:
            save_te_diagnostics(
                fit_te,
                valid_te,
                y[fit_idx],
                feature_columns,
                class_names,
                mapping,
                output_dir,
            )

        for spec in specs:
            experiment_dir = output_dir / spec.name
            fold_dir = experiment_dir / "folds"
            experiment_dir.mkdir(parents=True, exist_ok=True)
            fold_dir.mkdir(parents=True, exist_ok=True)
            config = {
                "experiment": spec.name,
                "te_groups": list(spec.te_groups),
                "drop_class": spec.drop_class,
                "drop_class_groups": (
                    list(spec.drop_class_groups)
                    if spec.drop_class_groups is not None
                    else None
                ),
                "feature_columns": feature_columns,
                "outer_n_splits": args.n_splits,
                "outer_seed": OUTER_SEED,
                "te_cv": TE_CV,
                "te_seed": TE_SEED,
                "te_smooth": "auto",
                "te_numeric": "direct",
                "model": "ft_transformer",
                "model_params": {"d_block": 128, "n_blocks": 2, "attention_n_heads": 8},
                "n_ens": args.n_ens,
                "n_epochs": args.n_epochs,
                "batch_size": 4096,
                "learning_rate": 0.001,
                "weight_decay": 0.00001,
                "numeric_scaler": "quantile",
                "num_embedding": "plr-lite",
                "model_seed": MODEL_SEED,
                "catstat_version": getattr(catstat, "__version__", "unknown"),
                "masamlp_version": getattr(masamlp, "__version__", "unknown"),
                "torch_version": torch.__version__,
            }
            config_json = json.dumps(config, sort_keys=True)
            (experiment_dir / "config.json").write_text(
                json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8"
            )
            fold_path = fold_dir / f"fold_{fold:02d}.npz"
            fold_started = time.time()
            if args.resume and fold_path.exists():
                cached = np.load(fold_path, allow_pickle=False)
                if not fold_cache_matches(
                    cached, valid_idx, train_ids, test_ids, config_json
                ):
                    raise ValueError(f"Fold cache does not match current run: {fold_path}")
                valid_proba = cached["valid_proba"].astype(np.float32)
                fold_test_proba = cached["test_proba"].astype(np.float32)
                source = "cache"
            else:
                fit_frame = model_frame(
                    fit_raw, fit_te, feature_columns, class_names, spec
                )
                valid_frame = model_frame(
                    valid_raw, valid_te, feature_columns, class_names, spec
                )
                test_frame = model_frame(
                    test_raw, test_te, feature_columns, class_names, spec
                )
                model = make_model(
                    categorical_columns,
                    n_ens=args.n_ens,
                    n_epochs=args.n_epochs,
                    device=args.device,
                )
                model.fit(fit_frame, y[fit_idx])
                valid_proba = reorder_model_probability(
                    model, model.predict_proba(valid_frame)
                )
                fold_test_proba = reorder_model_probability(
                    model, model.predict_proba(test_frame)
                )
                np.savez_compressed(
                    fold_path,
                    valid_idx=valid_idx,
                    valid_ids=train_ids[valid_idx],
                    test_ids=test_ids,
                    valid_proba=valid_proba,
                    test_proba=fold_test_proba,
                    classes=classes,
                    config_json=np.asarray(config_json),
                )
                source = "trained"
                del model, fit_frame, valid_frame, test_frame
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            state[spec.name]["oof"][valid_idx] = valid_proba
            state[spec.name]["test_sum"] += fold_test_proba
            row = {
                "fold": fold,
                "source": source,
                "raw_balanced_accuracy": fast_balanced_accuracy(
                    y[valid_idx], valid_proba.argmax(axis=1)
                ),
                "raw_logloss": log_loss(
                    y[valid_idx], valid_proba, labels=np.arange(3)
                ),
                "seconds": time.time() - fold_started,
            }
            state[spec.name]["fold_rows"].append(row)
            print(json.dumps({"experiment": spec.name, **row}), flush=True)
            del valid_proba, fold_test_proba
            gc.collect()

        del fit_raw, valid_raw, test_raw, fit_te, valid_te, test_te
        gc.collect()

    if np.any(fold_id < 0):
        raise RuntimeError("OOF fold_id is incomplete")
    prior = np.bincount(y, minlength=3).astype(np.float64) / len(y)
    consolidated = []
    for spec in specs:
        raw_oof = state[spec.name]["oof"]
        if np.any(raw_oof.sum(axis=1) <= 0):
            raise RuntimeError(f"OOF matrix is incomplete for {spec.name}")
        oof = normalize_probability(raw_oof).astype(np.float32)
        test_proba = normalize_probability(
            state[spec.name]["test_sum"] / args.n_splits
        ).astype(np.float32)
        experiment_dir = output_dir / spec.name
        config = json.loads((experiment_dir / "config.json").read_text(encoding="utf-8"))
        consolidated.extend(
            evaluate_and_save_full_experiment(
                spec,
                args,
                y,
                fold_id,
                oof,
                test_proba,
                prior,
                sample,
                label_encoder,
                classes,
                train_ids,
                test_ids,
                state[spec.name]["fold_rows"],
                experiment_dir,
                config,
            )
        )
    consolidated_df = pd.DataFrame(consolidated)
    consolidated_df.to_csv(output_dir / "full_cv_comparison.csv", index=False)
    print("\nFull CV comparison")
    print(consolidated_df.to_string(index=False))


## 7. 실행

기본 `lofo-screen` 결과에서 `lofo_importance`가 양수인 그룹을 우선 후보로 보되, 작은 차이는 seed noise일 수 있으므로 상위 6·8·10개 누적 후보를 정식 CV에서 확인하는 것이 안전합니다.


In [ ]:
def main() -> None:
    args = parse_args()
    if args.n_splits < 2 or args.n_ens < 1 or args.n_epochs < 1:
        raise ValueError("n-splits >= 2, n-ens >= 1, and n-epochs >= 1 are required")
    if args.screen_n_ens < 1 or args.screen_epochs < 1:
        raise ValueError("screen-n-ens and screen-epochs must be >= 1")
    if not 0 < args.screen_valid_size < 1:
        raise ValueError("screen-valid-size must be between 0 and 1")

    import torch

    if args.device.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. Use a Kaggle GPU session.")

    data_dir = resolve_data_dir(args.data_dir)
    output_dir = resolve_output_dir(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    train = pd.read_csv(data_dir / "train.csv")
    test = pd.read_csv(data_dir / "test.csv")
    sample_path = data_dir / "sample_submission.csv"
    sample = (
        pd.read_csv(sample_path)
        if sample_path.exists()
        else test[[ID_COL]].assign(**{TARGET: ""})
    )
    feature_columns = [column for column in test.columns if column != ID_COL]
    categorical_columns = [
        column
        for column in feature_columns
        if not pd.api.types.is_numeric_dtype(train[column])
    ]
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(train[TARGET]).astype(np.int8)
    classes = label_encoder.classes_.astype(str)
    raw_train = train[feature_columns]
    raw_test = test[feature_columns]

    run_manifest = {
        "run_mode": args.run_mode,
        "data_rows": [len(train), len(test)],
        "feature_columns": feature_columns,
        "categorical_columns": categorical_columns,
        "classes": classes.tolist(),
        "reference_class": args.reference_class,
        "screening_is_final_evidence": False,
    }
    (output_dir / "run_manifest.json").write_text(
        json.dumps(run_manifest, indent=2, ensure_ascii=False), encoding="utf-8"
    )

    if args.run_mode in {"audit-only", "lofo-screen", "candidate-screen"}:
        run_screening(
            args,
            raw_train,
            y,
            feature_columns,
            categorical_columns,
            classes.tolist(),
            output_dir,
        )
    else:
        run_full_cv(
            args,
            train,
            test,
            sample,
            raw_train,
            raw_test,
            y,
            label_encoder,
            feature_columns,
            categorical_columns,
            classes,
            output_dir,
        )


if __name__ == "__main__":
    main()


---

## 결과 해석 기준

1. `te_probability_sum_audit.csv`의 `max_abs_sum_error`가 모든 그룹에서 매우 작다면 3개 class TE 중 하나는 수학적으로 거의 중복입니다.
2. `te_classwise_correlation.csv`는 팀 논의를 위한 진단표이지만, correlation만으로 제거 여부를 결정하지 않습니다.
3. `lofo_screening.csv`에서 `lofo_importance > 0`이면 해당 그룹 제거로 BA가 하락한 것입니다.
4. screening의 작은 차이는 seed·짧은 epoch의 영향일 수 있습니다. 후보 구성은 같은 7-fold에서 `full-compare` 또는 `selected-full`로 확인합니다.
5. 최종 판단은 fixed inverse-prior Balanced Accuracy, 세 클래스 recall, OOF LogLoss, 학습시간을 함께 봅니다.

권장 순서:

1. 기본 `lofo-screen` 실행
2. `drop_at_risk39`의 타당성과 유용한 TE 그룹 확인
3. `full-compare`로 52개 vs 39개 정식 비교
4. 상위 6·8·10개 그룹을 각각 `selected-full`로 확인
5. 가장 단순하면서 fold별 성능이 안정적인 구성을 최종 FTT로 채택
